###Run Shared Libraries

In [0]:
%run ../Misc/SharedLibraries

In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimparty"

###Read Bronze table

In [0]:
partiesDf = spark.table("bronze.parties")
partyaddressDf = spark.table("bronze.partyaddress")

###Build Dimension/Fact table

In [0]:
dimPartyDf = partiesDf.join(
    partyaddressDf, partiesDf.PartyId == partyaddressDf.PartyNumber, "left"
    ).filter(partiesDf.RecordId.isNotNull()
    ).select(
        partiesDf.PartyId,
        F.trim(partiesDf.PartyName).alias("PartyName"),
        F.when(partiesDf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(partiesDf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
        F.from_utc_timestamp(partiesDf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
        partiesDf.PartyAddressCode.alias("PartyAddressCode"),
        F.from_utc_timestamp(partiesDf.EstablishedDate,'CST').alias("EstablishedDate"),
        F.trim(partiesDf.PartyEmailId).alias("PartyEmailId"),
        F.trim(partiesDf.PartyContactNumber).alias("PartyContactNumber"),
        partiesDf.RecordId.alias("PartyRecordId"),
        F.trim(partiesDf.TaxId).alias("TaxId"),
        F.trim(partyaddressDf.Address).alias("Address"),
        F.trim(partyaddressDf.City).alias("City"),
        F.trim(partyaddressDf.State).alias("State"),
        F.trim(partyaddressDf.Country).alias("Country"),
        F.trim(partyaddressDf.Region).alias("Region"),
        F.from_utc_timestamp(partyaddressDf.ValidFrom,'CST').alias("ValidFrom"),
        F.when(partyaddressDf.ValidTo.isNull(), F.lit("1900-01-01")).otherwise(partyaddressDf.ValidTo).cast("timestamp").alias("ValidTo"),
        partyaddressDf.RecordId.alias("PartyAddressRecordId")
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("PartyHashKey", F.xxhash64("PartyRecordId")
    )

display(dimPartyDf)

###Final dataframe

In [0]:
df_final = dimPartyDf

## Write to Silver Schema

In [0]:
saveDeltaTableToCatalog(df_final,"silver",Entity)

In [0]:
dimPartyDf.printSchema()

In [0]:
partyaddressDf.printSchema()